<a href="https://colab.research.google.com/github/Goseungeun/2026_BigData_Analyst/blob/main/Part4/Test_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1

In [10]:
# Q1
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/student_assessment.csv')
df = df.dropna()


id = df['id_assessment'].value_counts().idxmax()
cond = df['id_assessment'] == id
df = df[cond]
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df['score'] = scaler.fit_transform(df[['score']])
print(round(df['score'].max(),3))


2.183


In [13]:
# Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/stock_market.csv')

df_corr = df.corr()['close'].abs()
col = df_corr.loc['DE1':'DE77'].idxmax()

print(round(df[col].mean(),4))

-0.0004


In [16]:
# Q3
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/air_quality.csv')

q1 = df['CO2'].quantile(0.25)
q3 = df['CO2'].quantile(0.75)
IQR = q3-q1

cond1 = df['CO2'] < q1 - 1.5*IQR
cond2 = df['CO2'] > q3 + 1.5*IQR

df = df[cond1|cond2]
print(len(df))

304


## Part 2

In [35]:
import pandas as pd
train = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/mart_train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/mart_test.csv')

print(train.shape)
print(test.shape)

print(train.info())

print(train.isnull().sum().sum())
print(test.isnull().sum().sum())

cols = train.columns[train.dtypes=='object'].tolist()

for col in cols:
  train_nu = train[col].nunique()
  test_nu = test[col].nunique()
  if train_nu != test_nu:
    print(col,'not same')

  else:
    print(col,'num : ', train_nu)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

target = train.pop('total')

train = pd.get_dummies(train)
test = pd.get_dummies(test)

x_train,x_val,y_train,y_val= train_test_split(train,target,test_size=0.2,random_state=42)

rf = RandomForestRegressor()
rf.fit(x_train,y_train)
val_pred = rf.predict(x_val)
print(root_mean_squared_error(y_val,val_pred))

pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv('result.csv',index=False)

print(pd.read_csv('result.csv').head())
print(pd.read_csv('result.csv').shape)

(700, 10)
(300, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   branch          700 non-null    object 
 1   city            700 non-null    object 
 2   customer_type   700 non-null    object 
 3   gender          700 non-null    object 
 4   product_line    700 non-null    object 
 5   total           700 non-null    float64
 6   payment_method  700 non-null    object 
 7   rating          700 non-null    float64
 8   time_of_day     700 non-null    object 
 9   day_name        700 non-null    object 
dtypes: float64(2), object(8)
memory usage: 54.8+ KB
None
0
0
branch num :  3
city num :  3
customer_type num :  2
gender num :  2
product_line num :  6
payment_method num :  3
time_of_day num :  3
day_name num :  7
409095.4130596211
         pred
0  434127.015
1  535158.855
2  286336.890
3  576382.905
4  406523.250
(300, 1)


## Part 3

In [52]:
# Q1
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/clam.csv')

train = df.iloc[:210]
test = df.iloc[210:]

from statsmodels.formula.api import logit
import numpy as np
# 1-1
model = logit('gender ~weight',data = train).fit()
print(round(np.exp(model.params['weight']),4)) # 1.0047

# 1-2
model2 = logit('gender ~weight+age+length+diameter+height',data = train).fit()
print(round(-2*model2.llf,2)) #286.93

# 1-3
pred = model.predict(test)
pred = (pred>0.5).astype(int)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test['gender'],pred)
print(round(1-accuracy,3))

Optimization terminated successfully.
         Current function value: 0.690045
         Iterations 4
1.0047
Optimization terminated successfully.
         Current function value: 0.683173
         Iterations 4
286.93
0.478


In [61]:
# Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/system_cpu.csv')
print(df.head())

# 2-1
df_corr = df.corr()['ERP']
print(df_corr) #0.882

# 2-2
from statsmodels.formula.api import ols
cond = df['CPU'] < 100
df_cpu = df[cond]

model_c = ols('ERP~Feature1+Feature2+Feature3+CPU',data=df_cpu).fit()
print(round(model_c.rsquared,3)) # 0.755

# 2-3
print(model_c.summary())
print(round(model_c.pvalues['Feature1'],3)) # 0.684

     ERP  Feature1  Feature2  Feature3    CPU
0   30.6     235.1      44.5      44.0  112.3
1   40.3      36.6      46.4      36.1   58.6
2   57.7      52.2      66.5       2.0   55.3
3  128.3     196.2      59.8      57.4  103.2
4   80.3      75.2      59.6      58.2  104.1
ERP         1.000000
Feature1   -0.053848
Feature2    0.092432
Feature3    0.882194
CPU         0.092455
Name: ERP, dtype: float64
0.755
                            OLS Regression Results                            
Dep. Variable:                    ERP   R-squared:                       0.755
Model:                            OLS   Adj. R-squared:                  0.736
Method:                 Least Squares   F-statistic:                     39.30
Date:                Fri, 19 Jun 2026   Prob (F-statistic):           5.36e-15
Time:                        06:17:31   Log-Likelihood:                -260.40
No. Observations:                  56   AIC:                             530.8
Df Residuals:                     